# Data Pipeline Preprocessing

This notebook demonstrates how to build a robust **Scikit-Learn Preprocessing Pipeline** for the ShipmentSure dataset.

| Component | Purpose |
|---|---|
| `SimpleImputer` | Handles any missing values (median for numerical, most frequent for categorical) |
| `StandardScaler` | Normalises numerical features so they share a common scale |
| `OneHotEncoder` | Converts categorical text features into binary dummy variables |
| `ColumnTransformer` | Applies the correct preprocessing steps to the correct columns simultaneously |

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

sns.set_theme(style="whitegrid")

## 2. Load Dataset

In [ ]:
df = pd.read_csv("Train.csv")

print("Dataset Loaded Successfully")
print("Shape :", df.shape)
display(df.sample(5))

## 3. Extract Features (X) and Target (y)

In [ ]:
def prepare_features(dataframe):
    temp_df = dataframe.copy()
    
    # Remove identifier
    if "ID" in temp_df.columns:
        temp_df.drop("ID", axis=1, inplace=True)
    
    target = "Reached.on.Time_Y.N"
    
    X_data = temp_df.drop(columns=[target])
    y_data = temp_df[target]
    
    return X_data, y_data

X, y = prepare_features(df)

print("Feature Columns:", X.columns.tolist())

## 4. Identify Feature Types

In [ ]:
num_cols = [col for col in X.columns if X[col].dtype != 'object']
cat_cols = [col for col in X.columns if X[col].dtype == 'object']

print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)

## 5. Build Pipelines

We create separate processing pipelines for numerical and categorical data.

In [ ]:
# Numerical Pipeline: Fill missing values with median -> Scale data
numeric_pipe = Pipeline([
    ("median_fill", SimpleImputer(strategy="median")),
    ("scale_values", StandardScaler())
])

# Categorical Pipeline: Fill missing values with mode -> One-Hot Encode
categorical_pipe = Pipeline([
    ("freq_fill", SimpleImputer(strategy="most_frequent")),
    ("encode_cat", OneHotEncoder(handle_unknown="ignore"))
])

### 5.1 Combine into `ColumnTransformer`

In [ ]:
shipment_preprocessor = ColumnTransformer(
    transformers=[
        ("numerical_processing", numeric_pipe, num_cols),
        ("categorical_processing", categorical_pipe, cat_cols)
    ]
)

shipment_preprocessor

## 6. Execute Pipeline

In [ ]:
X_processed = shipment_preprocessor.fit_transform(X)

print("Transformed Feature Shape:", X_processed.shape)

## 7. Target Variable Overview

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(x=y)
plt.title("On-Time Delivery Distribution")
plt.tight_layout()
plt.show()

In [ ]:
warehouse_summary = df.groupby("Warehouse_block")["Reached.on.Time_Y.N"].mean()

warehouse_summary.plot(kind="bar", figsize=(6, 4))
plt.title("Average On-Time Delivery by Warehouse")
plt.ylabel("Delivery Rate")
plt.tight_layout()
plt.show()